In [ ]:
import os

import dotenv
import requests

dotenv.load_dotenv(".env")


HF_TOKEN = os.environ.get("HF_API_KEY")
API_URL = "https://router.huggingface.co/hf-inference/models/MoritzLaurer/ModernBERT-large-zeroshot-v2.0"

headers = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json",
}


In [ ]:
def check_destructive_content(text: str, threshold: float = 0.6):
    """
    Проверяет текст на деструктивный контент через HF Inference API.
    Возвращает словарь с флагами и вероятностями.
    """
    # Категории для проверки (формулировки важны для точности!)
    candidate_labels = [
        "насилие и жестокость",
        "наркотики и запрещенные вещества",
        "алкоголь и курение",
        "ЛГБТ и нетрадиционные отношения",
        "ненормативная лексика",
        "призывы к суициду или самоповреждению"
    ]

    payload = {
        "inputs": text,
        "parameters": {
            "candidate_labels": candidate_labels,
            "multi_label": True,
            "hypothesis_template": "Этот текст содержит {}.",
        },
    }

    response = requests.post(API_URL, headers=headers, json=payload)

    if response.status_code != 200:
        raise Exception(f"API Error: {response.status_code} - {response.text}")

    result = response.json()

    # Обработка результатов
    detected = []
    scores = {}

    for label, score in zip(result["labels"], result["scores"]):
        scores[label] = score
        if score > threshold:
            detected.append(label)

    return {
        "is_destructive": len(detected) > 0,
        "detected_categories": detected,
        "scores": scores,
    }


In [ ]:
transcript = "Вчера мы выпили пару бутылок вина и закурили сигареты..."

result = check_destructive_content(transcript)
print(f"Деструктивный контент: {result['is_destructive']}")
print(f"Обнаружено: {result['detected_categories']}")
print(f"Вероятности: {result['scores']}")